In [7]:
import pandas as pd

# =========================
# 1. LOAD
# =========================
def load_data(paths):
    return {name: pd.read_csv(path) for name, path in paths.items()}


# =========================
# 2. NORMALIZAÇÃO
# =========================
def normalize_dates(df):
    if 'data_avaliacao' in df.columns:
        df['data_avaliacao'] = pd.to_datetime(df['data_avaliacao'], errors='coerce')
    return df


# =========================
# 3. PIVOT (lado → colunas)
# =========================
def pivot_lado(df, prefix):
    if 'lado' not in df.columns:
        return df

    id_cols = ['id_ficha', 'data_avaliacao']
    value_cols = [c for c in df.columns if c not in id_cols + ['lado']]

    df_pivot = df.pivot_table(
        index=id_cols,
        columns='lado',
        values=value_cols,
        aggfunc='first'
    )

    df_pivot.columns = [
        f"{prefix}_{col}_{lado}".lower()
        for col, lado in df_pivot.columns
    ]

    return df_pivot.reset_index()


# =========================
# 4. PROCESSAMENTO
# =========================
def build_dataset(data):

    # ---------
    # Base temporal (define o acompanhamento)
    # ---------
    base = normalize_dates(data['pontuacao']).copy()

    # garantir granularidade única
    base = base.drop_duplicates(subset=['id_ficha', 'data_avaliacao'])

    # ---------
    # Tabelas temporais (com data_avaliacao)
    # ---------
    face = pivot_lado(normalize_dates(data['face']), 'face')
    ms = pivot_lado(normalize_dates(data['membro_superior']), 'ms')
    mi = pivot_lado(normalize_dates(data['membro_inferior']), 'mi')
    mao = pivot_lado(normalize_dates(data['avaliacao_sensitiva_mao']), 'mao')
    pe = pivot_lado(normalize_dates(data['avaliacao_sensitiva_pe']), 'pe')
    queixas = normalize_dates(data['queixas'])

    # remover PKs desnecessárias
    queixas = queixas.drop(columns=['id_queixas'], errors='ignore')

    temporais = [face, ms, mi, mao, pe, queixas]

    # ---------
    # Merge temporal (CORRETO)
    # ---------
    df = base.copy()

    for temp_df in temporais:
        df = pd.merge(
            df,
            temp_df,
            on=['id_ficha', 'data_avaliacao'],
            how='left'
        )

    # ---------
    # Dimensões (sem data)
    # ---------
    ficha = data['ficha']
    paciente = data['paciente']

    df = df.merge(ficha, on='id_ficha', how='left')
    df = df.merge(paciente, on='id_paciente', how='left')

    return df


# =========================
# 5. EXECUÇÃO
# =========================
paths = {
    "paciente": "dataset/dataset-hof-2026 - Paciente.csv",
    "ficha": "dataset/dataset-hof-2026 - Ficha.csv",
    "face": "dataset/dataset-hof-2026 - Face.csv",
    "membro_superior": "dataset/dataset-hof-2026 - Membro superior.csv",
    "membro_inferior": "dataset/dataset-hof-2026 - Membro inferior.csv",
    "avaliacao_sensitiva_mao": "dataset/dataset-hof-2026 - Avaliação sensitiva mão.csv",
    "avaliacao_sensitiva_pe": "dataset/dataset-hof-2026 - Avaliação sensitiva pé.csv",
    "queixas": "dataset/dataset-hof-2026 - Queixas.csv",
    "pontuacao": "dataset/dataset-hof-2026 - Pontuação.csv"
}

data = load_data(paths)

df_final = build_dataset(data)

print(df_final.shape)
df_final.head()

(648, 113)


,id_pontuacao,id_ficha,data_avaliacao,olho_direito,olho_esquerdo,mao_direita,mao_esquerda,pe_direito,pe_esquerdo,maior_grau,...,municipio,uf,classificacao_operacional,patologias_associadas,medicacao,data_inicio_tratamento,data_termino_tratamento,observacoes,sexo,data_nasc
0,1,1,2023-08-03,0,0,2,0,1,0,2,...,Trindade,PE,MB,"HAS, Asma","Losartana, Hidroclorotiazida, Prednisona",2021,2022,"1° Tratamento 2020/2021, 2° Tratamento 2021/2022",Masculino,25/01/1989
1,2,2,2023-01-09,1,0,2,0,2,2,2,...,Jaboatão dos Guararapes,PE,MB,Sem registro,Sem registro,01/2023,Sem registro,1º Tratamento no município,Masculino,22/02/1973
2,3,3,NaT,0,0,0,0,0,0,0,...,Amaraji,PE,MB,Sem registro,Sem registro,Sem registro,Sem registro,1º Tratamento: 2024,Feminino,27/11/2011
3,4,4,NaT,1,1,0,0,2,2,2,...,Paulista,PE,MB,HAS,"Press plus, Spasmex",2022,2023,NaN,Feminino,01/04/1948
4,4,4,NaT,1,1,0,0,2,2,2,...,Paulista,PE,MB,HAS,"Press plus, Spasmex",2022,2023,NaN,Feminino,01/04/1948
